# SBERT Sentence Analysis

## 1. Preparations

### 1.1 Read Data
This new data is obtained after some further anonymization, and by putting previously incorrectly split sentences together, and removed duplicates

In [1]:
import pandas as pd
# read the sentence data
df = pd.read_excel("/workspace/mijnidbcoachnlp/data/analysis_data/translated_sentence_data_new.xlsx")
# check the head
df.head()

,Message_ID,Sentence,Clean_Sentence,Sentence_ID,Translated_Sentence
0,3,Ik ben 2 weken geleden met spoed opgenomen in ...,Ik ben 2 weken geleden met spoed opgenomen in ...,1,I was rushed into the [PRESSION] two weeks ago...
1,3,"Ik kreeg acuut een pijnlijke druk op de borst,...","Ik kreeg acuut een pijnlijke druk op de borst,...",2,I was acutely under a painful pressure on the ...
2,3,Het begon 1 uur na het avondeten.,Het begon 1 uur na het avondeten.,3,It started 1 hour after dinner.
3,3,"Ik had al de hele dag migraine, had dus ook we...","Ik had al de hele dag migraine, had dus ook we...",4,"I had migraines all day, so I hadn't eaten much."
4,3,"Ik werd heel erg misselijk, braakneigingen, du...","Ik werd heel erg misselijk, braakneigingen, du...",5,"I got very nauseous, vomiting, dizzy and rejuv..."


In [2]:
# clean sentences as inputs
sentences = df["Clean_Sentence"].to_list()

In [3]:
# print the size of the data
len(df)
# There are 41143 unique sentences 

41143

### 1.2. Import the list of stopwords

In [4]:
### Importing the list of Dutch stopwords (note that there are customized dutch words in there)

with open('/workspace/mijnidbcoachnlp/data/analysis_data/stopwords.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

extra_list = [
    'maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag', 'zaterdag', 'zondag',
    'januari', 'februari', 'maart', 'april', 'mei', 'juni', 'juli', 'augustus', 'september', 'oktober', 'november', 'december',
    'jan', 'feb', 'mrt', 'apr', 'mei', 'jun', 'jul', 'aug', 'sep', 'okt', 'nov', 'dec', "jl", "weken", "week", "dagen", "dag", "mg", "coach", "mijnibdcoach", "dr", "uur", "dgs"
]

dutch_stopwords.extend(extra_list)

### 1.4. Embed the lists of sentences with sentence-transformer multilingual-v1

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

# load the embedding model
embedding_model = SentenceTransformer("distiluse-base-multilingual-cased-v1")

# run the following for embeddings if no pre-saved embeddings can be loaded
embeddings = embedding_model.encode(sentences, show_progress_bar=True)
# save the pre-calculated embeddings 
np.save('/workspace/mijnidbcoachnlp/data/embeddings/embeddings_st_v1_new_input.npy', embeddings)

/workspace/mijnidbcoachnlp/venv/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2025-02-09 21:48:42.827119: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Batches:   0%|          | 0/1286 [00:00<?, ?it/s]

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np
# load pre-saved embeddings if you have, otherwise calculate them using the commented-out codes
embedding_model = SentenceTransformer("distiluse-base-multilingual-cased-v1")
embeddings = np.load("/workspace/mijnidbcoachnlp/data/embeddings/embeddings_st_v1_new_input.npy")

### 1.5 functions to pre-configure burtopic model

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer_model = CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{3,}\b')

In [21]:
# function to build the model with different selection of clustering method
# build the BERTopic model
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer


def tune_umap(n_neighbors, n_components, min_dist):
    umap_model = UMAP(n_neighbors=n_neighbors, n_components=n_components, min_dist=min_dist, metric='cosine', random_state=42)
    return umap_model

def tune_hdb(min_cluster_size, min_samples, cluster_selection_epsilon):
    hdb_model = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples, cluster_selection_epsilon=cluster_selection_epsilon, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
    return hdb_model

def build_bertopic(embedding_model, umap_model, cluster_model, vectorizer_model, embeddings, docs):
    # Initialize BERTopic with UMAP and HDBSCAN models
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        top_n_words=10,
        verbose=True
    )

    # Fit-transform to get document topics and probabilities
    topics, probs = topic_model.fit_transform(docs, embeddings)
    

    # Return only essential results
    return topic_model

## 2. Build BERTopic Model

In [9]:
# disable parallelism to avoid some warnings
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [10]:
# build the model with the preconfigured settings in 1.5
# use a empirical set of parameters 
umap_model = tune_umap(15, 5, 0.05)
hdb_model = tune_hdb(20, 10, 0.0)
topic_model = build_bertopic(embedding_model=embedding_model, umap_model=umap_model, cluster_model=hdb_model, vectorizer_model=vectorizer_model, embeddings=embeddings, docs=sentences)

2025-02-09 21:53:08,152 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-09 21:54:34,850 - BERTopic - Dimensionality - Completed ✓
2025-02-09 21:54:34,854 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-09 21:54:39,183 - BERTopic - Cluster - Completed ✓
2025-02-09 21:54:39,199 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-09 21:54:39,989 - BERTopic - Representation - Completed ✓


In [11]:
# examine topic information
pd.set_option("max_colwidth", 200) # adjust column width 
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,15586,-1_vraag_afspraak_medicatie_krijg,"[vraag, afspraak, medicatie, krijg, last, vragen, klachten, gaat, volgende, laatste]","[Zouden jullie eens willen kijken of de uitslag bekend is en uitslag doorsturen a.u.b. Indien de ontsteking goed dalende is dan kan ik een plan maken om terug op te starten met werken., Mijn vraa..."
1,0,3404,0_bloed_prikken_bloedprikken_ontlasting,"[bloed, prikken, bloedprikken, ontlasting, slijm, bloeduitslagen, laten, geprikt, ingeleverd, inleveren]","[Ik heb 19 november voor het laatst mijn bloed laten prikken., Ok, dan ga ik deze week nog bloed laten prikken., Ik heb maandag bloed laten prikken.]"
2,1,1470,1_huisarts_arts_dokter_mdl,"[huisarts, arts, dokter, mdl, ziekenhuis, contact, verpleegkundige, reumatoloog, medische, gynaecoloog]","[ik ben net bij de huisarts geweest., Ik ben net bij de huisarts geweest., huisarts heeft ca.]"
3,2,706,2_test_testen_proberen_quanton,"[test, testen, proberen, quanton, cal, zelftest, gedaan, geprobeerd, thuis, calprotectine]","[Ook de test voor thiopurinewaarden., Ik heb dus geen test kunnen doen., Wanneer moet ik z’n test doen?]"
4,3,490,3_recept_apotheek_nieuw_sturen,"[recept, apotheek, nieuw, sturen, doorsturen, herhaalrecept, faxen, uitschrijven, questran, purinethol]","[Zou u een nieuw recept naar de apotheek in het [ZIEKENHUIS] kunnen sturen?, Kunt u een nieuw recept naar apotheek voor mij sturen?, Kunnen jullie a.u.b. een nieuw recept naar de apotheek sturen.]"
...,...,...,...,...,...
252,251,20,251_nakijken_kijken_zicht_tijden,"[nakijken, kijken, zicht, tijden, bekijken, checken, toesturen, zekerheid, uitslag, ]","[Wil één van jullie dit voor mij nakijken?, Kunt u dat voor me nakijken aub?, Wil je dit voor me nakijken?]"
253,252,20,252_invloed_gevolgen_vruchtbaarheid_effect,"[invloed, gevolgen, vruchtbaarheid, effect, hoeverre, slank, remicadeinfuus, mislukte, impact, leverwaardes]","[Heeft dit invloed op de uitslag?, Kan dat invloed hebben?, Heeft dat nog invloed op mijn ontlasting?]"
254,253,20,253_afwezigheid_eigelijk_uitgerekend_verhuizen,"[afwezigheid, eigelijk, uitgerekend, verhuizen, afspraakbevestiging, sticker, biopt, gevaccineerd, darmonderzoek, min]","[Toen mijn oproep begin juni kwam heb ik ook gelijk mijn afspraken gemaakt en zo ben ik dus ook al volledig gevaccineerd., Zoals jullie weten had ik eigelijk 30 juni een afspraak voor een kleine c..."
255,254,20,254_mogelijk_mogelijkheid__,"[mogelijk, mogelijkheid, , , , , , , , ]","[Hoe is dit mogelijk?, Is dit mogelijk?, Is dit mogelijk?]"


In [12]:
# Calculate the coherence score of the model
# Note: to evaluate with coherence score, the n_gram of the vectorizer_model has to be (1, 1)
def get_top_words(topic_model):
    topics = topic_model.get_topics()
    top_words = [[word for word, _ in topic[:10]] for topic in topics.values() if topic]  # Top 10 words per topic
    return top_words

In [13]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

# Create a dictionary representation of the words in topics
top_words = get_top_words(topic_model)
tokenized_texts = [doc.split() for doc in sentences]  # Tokenizing by splitting on spaces
dictionary = Dictionary(tokenized_texts)

# Create a coherence model
coherence_model = CoherenceModel(topics=top_words, 
                                texts=tokenized_texts,
                                dictionary=dictionary, 
                                coherence='c_v')

# Compute coherence score
coherence_score = coherence_model.get_coherence()
print(f"Coherence Score: {coherence_score}")


Coherence Score: 0.35337282532042696


In [14]:
# reduce outliers of this topic model and see what changes to the coherence score
topics = topic_model.topics_
new_topics = topic_model.reduce_outliers(sentences, topics)

100%|██████████| 16/16 [00:01<00:00,  8.59it/s]


In [15]:
topic_model.update_topics(vectorizer_model=vectorizer_model, docs=sentences, topics=new_topics)

2025-02-09 21:56:36,898 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [16]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,249,-1_beangstigend_azethoprine_adhd_nieuwsberichten,"[beangstigend, azethoprine, adhd, nieuwsberichten, wijzer, wachttijden, vriendleijke, vakjes, tijdsbestek, tijdelijks]","[Zouden jullie eens willen kijken of de uitslag bekend is en uitslag doorsturen a.u.b. Indien de ontsteking goed dalende is dan kan ik een plan maken om terug op te starten met werken., Mijn vraa..."
1,0,3549,0_bloed_prikken_bloedprikken_ontlasting,"[bloed, prikken, bloedprikken, ontlasting, slijm, laten, bloeduitslagen, geprikt, ingeleverd, inleveren]","[Ik heb 19 november voor het laatst mijn bloed laten prikken., Ok, dan ga ik deze week nog bloed laten prikken., Ik heb maandag bloed laten prikken.]"
2,1,1605,1_huisarts_arts_dokter_mdl,"[huisarts, arts, dokter, mdl, ziekenhuis, contact, reumatoloog, verpleegkundige, gynaecoloog, medische]","[ik ben net bij de huisarts geweest., Ik ben net bij de huisarts geweest., huisarts heeft ca.]"
3,2,758,2_test_testen_proberen_quanton,"[test, testen, proberen, quanton, cal, gedaan, zelftest, geprobeerd, thuis, calprotectine]","[Ook de test voor thiopurinewaarden., Ik heb dus geen test kunnen doen., Wanneer moet ik z’n test doen?]"
4,3,734,3_recept_apotheek_nieuw_sturen,"[recept, apotheek, nieuw, sturen, uitschrijven, herhaalrecept, doorsturen, purinethol, faxen, gestuurd]","[Zou u een nieuw recept naar de apotheek in het [ZIEKENHUIS] kunnen sturen?, Kunt u een nieuw recept naar apotheek voor mij sturen?, Kunnen jullie a.u.b. een nieuw recept naar de apotheek sturen.]"
...,...,...,...,...,...
252,251,44,251_nakijken_checken_zekerheid_dagenlang,"[nakijken, checken, zekerheid, dagenlang, ongeluk, ongemakkelijk, ingesteld, relevante, algeheel, kijken]","[Wil één van jullie dit voor mij nakijken?, Kunt u dat voor me nakijken aub?, Wil je dit voor me nakijken?]"
253,252,65,252_invloed_effect_gevolgen_vruchtbaarheid,"[invloed, effect, gevolgen, vruchtbaarheid, hoeverre, werking, metformine, diabetes, verband, medicijngebruik]","[Heeft dit invloed op de uitslag?, Kan dat invloed hebben?, Heeft dat nog invloed op mijn ontlasting?]"
254,253,72,253_afbouw_darmonderzoek_trouwens_volledig,"[afbouw, darmonderzoek, trouwens, volledig, uitgerekend, verhuizen, biopt, gepaard, gevaccineerd, erop]","[Toen mijn oproep begin juni kwam heb ik ook gelijk mijn afspraken gemaakt en zo ben ik dus ook al volledig gevaccineerd., Zoals jullie weten had ik eigelijk 30 juni een afspraak voor een kleine c..."
255,254,202,254_mogelijk_mogelijkheid_imuran_krijgen,"[mogelijk, mogelijkheid, imuran, krijgen, ophaal, indien, ontstekingswaarden, verkrijgen, schrijven, grieperig]","[Hoe is dit mogelijk?, Is dit mogelijk?, Is dit mogelijk?]"


In [17]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

# Create a dictionary representation of the words in topics
top_words = get_top_words(topic_model)

# Create a coherence model
coherence_model = CoherenceModel(topics=top_words, 
                                texts=tokenized_texts,
                                dictionary=dictionary, 
                                coherence='c_v')

# Compute coherence score
coherence_score = coherence_model.get_coherence()
print(f"Coherence Score of Model after Outlier Reduction: {coherence_score}")


Coherence Score of Model after Outlier Reduction: 0.3524355715851859


## 3. Fine-Tune the Model

In [34]:
# function to calculate the coherence score of a topic model 

from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

def calculate_c_v(topic_model):
    top_words = get_top_words(topic_model)
    coherence_model = CoherenceModel(topics=top_words, 
                                texts=tokenized_texts,
                                dictionary=dictionary, 
                                coherence='c_v')
    coherence_score = coherence_model.get_coherence()
    return coherence_score

### 3.1 Fine-Tune UMAP hyperparameters

In [19]:
import itertools

# Define parameter ranges
range_n_neighbors = [5, 10, 20, 35, 50, 75, 100]
range_n_components = [2, 3, 4, 5, 6]
range_min_dist = [0.0, 0.01, 0.05, 0.1]

# Generate all possible combinations of the parameters
umap_combinations = list(itertools.product(range_n_neighbors, range_n_components, range_min_dist))

In [30]:
import pandas as pd

def select_comb(n_iter, umap_combinations):
    # Randomly select `n_iter` combinations
    sample_combinations = random.sample(umap_combinations, n_iter)
    
    # Get the untried combinations by dropping the selected ones
    untried_combinations = set(umap_combinations) - set(sample_combinations)
    
    return sample_combinations, untried_combinations

In [35]:
import random
from joblib import Parallel, delayed

def tune_bertopic_umap_random(embedding_model, vectorizer_model, embeddings, sentences, umap_combinations, n_iter):
    """
    Randomized search over UMAP and HDBSCAN hyperparameters to find the best model based on coherence score.
    
    Args:
        embedding_model: The embedding model to use for topic modeling.
        vectorizer_model: The vectorizer model to use for tokenization.
        embeddings: The embeddings of the sentences.
        sentences: The sentences (documents) for topic modeling.
        n_iter: The number of random iterations to perform in the randomized search.
        
    Returns:
        best_topic_model: The best topic model based on coherence score.
        best_coherence_score: The best coherence score obtained during the search.
    """
    sample_combinations, untried_combinations = select_comb(n_iter, umap_combinations)
    best_coherence_score = -float('inf')
    best_topic_model = None

    # Randomized search loop
    for comb in sample_combinations:
        # Sample random UMAP parameters
        n_neighbors = comb[0]
        n_components = comb[1]
        min_dist = comb[2]
        # Tune UMAP and HDBSCAN
        umap_model = tune_umap(n_neighbors=n_neighbors, n_components=n_components, min_dist=min_dist)
        hdb_model = tune_hdb(min_cluster_size=20, min_samples=10, cluster_selection_epsilon=0.0)

        # Build topic model
        current_topic_model = build_bertopic(
            embedding_model=embedding_model, 
            umap_model=umap_model, 
            cluster_model=hdb_model, 
            vectorizer_model=vectorizer_model, 
            embeddings=embeddings, 
            docs=sentences
        )
        
        # Calculate coherence score for the current model
        current_coherence_score = calculate_c_v(current_topic_model)
        print(f"Coherence score: {current_coherence_score} for UMAP params: {n_neighbors}, {n_components}, {min_dist}")
        
        # Check if the current model has a better coherence score
        if current_coherence_score > best_coherence_score:
            best_coherence_score = current_coherence_score
            best_topic_model = current_topic_model
            best_combination = (n_neighbors, n_components, min_dist)
            
    return best_topic_model, best_coherence_score, best_combination, untried_combinations

# Example usage
best_model, best_score, best_combination, untried_combinations = tune_bertopic_umap_random(embedding_model=embedding_model, vectorizer_model=vectorizer_model, embeddings=embeddings, sentences=sentences, n_iter=20, umap_combinations=umap_combinations)
print("Best UMAP Parameters:", best_combination)
print("Best Coherence Score:", best_score)


2025-02-09 22:30:18,945 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-09 22:32:18,143 - BERTopic - Dimensionality - Completed ✓
2025-02-09 22:32:18,147 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-09 22:32:20,927 - BERTopic - Cluster - Completed ✓
2025-02-09 22:32:20,942 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-09 22:32:21,681 - BERTopic - Representation - Completed ✓
2025-02-09 22:33:02,705 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Coherence score: 0.36381295149064835 for UMAP params: 50, 5, 0.01


2025-02-09 22:34:46,402 - BERTopic - Dimensionality - Completed ✓
2025-02-09 22:34:46,405 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-09 22:34:47,980 - BERTopic - Cluster - Completed ✓
2025-02-09 22:34:47,994 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-09 22:34:48,730 - BERTopic - Representation - Completed ✓
2025-02-09 22:35:31,206 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Coherence score: 0.3664326881483675 for UMAP params: 50, 2, 0.01


2025-02-09 22:37:14,947 - BERTopic - Dimensionality - Completed ✓
2025-02-09 22:37:14,950 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-09 22:37:16,504 - BERTopic - Cluster - Completed ✓
2025-02-09 22:37:16,517 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-09 22:37:17,251 - BERTopic - Representation - Completed ✓
2025-02-09 22:38:00,640 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


...

Coherence score: 0.36853693977844776 for UMAP params: 100, 6, 0.05


2025-02-09 23:20:15,227 - BERTopic - Dimensionality - Completed ✓
2025-02-09 23:20:15,230 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-09 23:20:16,786 - BERTopic - Cluster - Completed ✓
2025-02-09 23:20:16,800 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-09 23:20:17,503 - BERTopic - Representation - Completed ✓
2025-02-09 23:20:57,672 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Coherence score: 0.3532781405835234 for UMAP params: 100, 2, 0.01


2025-02-09 23:22:08,169 - BERTopic - Dimensionality - Completed ✓
2025-02-09 23:22:08,172 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-09 23:22:11,645 - BERTopic - Cluster - Completed ✓
2025-02-09 23:22:11,659 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-09 23:22:12,416 - BERTopic - Representation - Completed ✓


Coherence score: 0.3560997064761888 for UMAP params: 20, 6, 0.05
Best UMAP Parameters: (100, 6, 0.05)
Best Coherence Score: 0.36853693977844776


Best UMAP Parameters: (100, 6, 0.05)
Best Coherence Score: 0.36853693977844776

In [37]:
untried_combinations

{(5, 2, 0.0),
 (5, 2, 0.01),
 (5, 2, 0.05),
 (5, 2, 0.1),
 (5, 3, 0.0),
 (5, 3, 0.01),
 (5, 3, 0.1),
 (5, 4, 0.0),
 (5, 4, 0.01),
 (5, 4, 0.05),
 (5, 4, 0.1),
 (5, 5, 0.0),
 (5, 5, 0.01),
 (5, 5, 0.05),
 (5, 5, 0.1),
 (5, 6, 0.0),
 (5, 6, 0.05),
 (10, 2, 0.0),
 (10, 2, 0.01),
 (10, 2, 0.05),
 (10, 2, 0.1),
 (10, 3, 0.0),
 (10, 3, 0.01),
 (10, 3, 0.05),
 (10, 3, 0.1),
 (10, 4, 0.0),
 (10, 4, 0.01),
 (10, 4, 0.05),
 (10, 4, 0.1),
 (10, 5, 0.0),
 (10, 5, 0.01),
 (10, 5, 0.05),
 (10, 5, 0.1),
 (10, 6, 0.0),
 (10, 6, 0.01),
 (10, 6, 0.05),
 (10, 6, 0.1),
 (20, 2, 0.0),
 (20, 2, 0.05),
 (20, 2, 0.1),
 (20, 3, 0.01),
 (20, 3, 0.05),
 (20, 3, 0.1),
 (20, 4, 0.0),
 (20, 4, 0.01),
 (20, 5, 0.0),
 (20, 5, 0.01),
 (20, 5, 0.05),
 (20, 5, 0.1),
 (20, 6, 0.0),
 (20, 6, 0.01),
 (20, 6, 0.1),
 (35, 2, 0.01),
 (35, 2, 0.05),
 (35, 2, 0.1),
 (35, 3, 0.0),
 (35, 3, 0.01),
 (35, 3, 0.05),
 (35, 3, 0.1),
 (35, 4, 0.0),
 (35, 4, 0.01),
 (35, 4, 0.05),
 (35, 4, 0.1),
 (35, 5, 0.0),
 (35, 5, 0.01),
 (35, 5, 0

In [38]:
best_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,16828,-1_afspraak_klachten_last_krijg,"[afspraak, klachten, last, krijg, gaat, keer, vandaag, gaan, goed, tijd]","[Ontlasting gaat ook veel beter en soms maar 1 keer per dag., Gisteren kreeg ik een brief waarin stond dat ik binnenkort gebeld ga worden door de MDL-arts en of ik mee wilde doen aan een onderzoek..."
1,0,3386,0_bloed_prikken_ontlasting_bloedprikken,"[bloed, prikken, ontlasting, bloedprikken, slijm, laten, bloeduitslagen, geprikt, ingeleverd, inleveren]","[Afgelopen maandag heb ik bloed laten prikken., Moet ik dan alsnog ook bloed laten prikken?, Ik heb ook bloed laten prikken en ontlasting.]"
2,1,3128,1_recept_apotheek_nieuw_medicatie,"[recept, apotheek, nieuw, medicatie, medicijnen, tabletten, sturen, herhaalrecept, bestellen, medicijn]","[Graag nieuw recept naar apotheek ., Graag recept naar Apotheek ., Nu is het recept nog niet binnen bij de apotheek.]"
3,2,1463,2_huisarts_arts_dokter_ziekenhuis,"[huisarts, arts, dokter, ziekenhuis, mdl, contact, reumatoloog, verpleegkundige, chirurg, medische]","[Ik ben net bij de huisarts geweest., Kan dit alles bij de huisarts?, In het ziekenhuis of de huisarts.]"
4,3,1250,3_gepland_afspraak_staat_endoscopie,"[gepland, afspraak, staat, endoscopie, staan, poli, eind, begin, infuus, vakantie]","[Dit staat op 24 juni a.s. om 10:25 gepland., Mijn 3de infuus staat gepland op 28 december., De endoscopie staat gepland op 23 mei om 15:20.]"
...,...,...,...,...,...
183,182,20,182_waarde_gedaald_leverwaarde_referentiewaarde,"[waarde, gedaald, leverwaarde, referentiewaarde, referentiewaarden, waardes, gemeten, toegevoegde, hoogste, war]","[De gemeten waarde is geen verassing en het gaat dan ook redelijk., Ervan uit gaande dat de uitslag klopt, is de waarde wel flink gedaald., De waarde in gezien te hebben lijkt het erop dat, alhoew..."
184,183,20,183_darmkanker_bevolkingsonderzoek_meegedaan_deelname,"[darmkanker, bevolkingsonderzoek, meegedaan, deelname, onlangs, uitnodiging, oproep, deel, landelijk, zelfafnametest]","[heer , Onlangs een brief ontvangen om aan een bevolkingsonderzoek darmkanker mee te doen., Ik heb een uitnodiging gekregen voor het bevolkingsonderzoek naar darmkanker., Verder heb ik een vooraan..."
185,184,20,184_hoor_graag_geboorte_tips,"[hoor, graag, geboorte, tips, , , , , , ]","[Ik hoor graag van jullie,, Ik hoor graag van jullie,, Ik hoor graag van jullie,]"
186,185,20,185_reactie_bedankt__,"[reactie, bedankt, , , , , , , , ]","[Bedankt voor je reactie., Bedankt voor je reactie., Bedankt voor je reactie.]"


In [39]:
# try another 20 iterations 
best_model, best_score, best_combination, untried_combinations = tune_bertopic_umap_random(embedding_model=embedding_model, vectorizer_model=vectorizer_model, embeddings=embeddings, sentences=sentences, n_iter=20, umap_combinations=untried_combinations)
print("Best UMAP Parameters:", best_combination)
print("Best Coherence Score:", best_score)

2025-02-10 13:40:07,036 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-10 13:41:12,554 - BERTopic - Dimensionality - Completed ✓
2025-02-10 13:41:12,561 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-10 13:41:18,345 - BERTopic - Cluster - Completed ✓
2025-02-10 13:41:18,380 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-10 13:41:20,166 - BERTopic - Representation - Completed ✓
2025-02-10 13:43:29,013 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Coherence score: 0.3484900899365496 for UMAP params: 5, 4, 0.01


2025-02-10 13:44:38,576 - BERTopic - Dimensionality - Completed ✓
2025-02-10 13:44:38,581 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-10 13:44:42,508 - BERTopic - Cluster - Completed ✓
2025-02-10 13:44:42,535 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-10 13:44:44,036 - BERTopic - Representation - Completed ✓
2025-02-10 13:46:33,226 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Coherence score: 0.344384247801974 for UMAP params: 5, 4, 0.05


2025-02-10 13:51:22,080 - BERTopic - Dimensionality - Completed ✓
2025-02-10 13:51:22,083 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-10 13:51:29,126 - BERTopic - Cluster - Completed ✓
2025-02-10 13:51:29,146 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-10 13:51:30,242 - BERTopic - Representation - Completed ✓
2025-02-10 13:52:33,478 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


...

Coherence score: 0.35405116879802223 for UMAP params: 35, 4, 0.01


2025-02-10 14:47:10,689 - BERTopic - Dimensionality - Completed ✓
2025-02-10 14:47:10,692 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-10 14:47:17,768 - BERTopic - Cluster - Completed ✓
2025-02-10 14:47:17,787 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-10 14:47:18,650 - BERTopic - Representation - Completed ✓
2025-02-10 14:48:11,266 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Coherence score: 0.3705792198430164 for UMAP params: 100, 6, 0.1


2025-02-10 14:49:00,680 - BERTopic - Dimensionality - Completed ✓
2025-02-10 14:49:00,684 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-10 14:49:03,020 - BERTopic - Cluster - Completed ✓
2025-02-10 14:49:03,040 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-10 14:49:04,160 - BERTopic - Representation - Completed ✓


Coherence score: 0.3493718472769868 for UMAP params: 5, 3, 0.01
Best UMAP Parameters: (100, 6, 0.1)
Best Coherence Score: 0.3705792198430164


In [42]:
# check the topic info
best_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,18560,-1_medicatie_klachten_afspraak_krijg,"[medicatie, klachten, afspraak, krijg, vraag, gaat, gaan, vandaag, last, tijd]",[De klysmas zijn nog niet gekomen .. ze verwachten pas eind volgende week.. dan heb ik niet genoeg voor de tussen tijd … de apotheek gaat bellen naar het ziekenhuis apotheek om een doosje te kunne...
1,0,3364,0_bloed_prikken_ontlasting_bloedprikken,"[bloed, prikken, ontlasting, bloedprikken, slijm, laten, bloeduitslagen, geprikt, ingeleverd, inleveren]","[Zij heeft bloed laten prikken., Twee weken geleden heb ik bloed laten prikken., Moet ik misschien bloed laten prikken?]"
2,1,2006,1_vakantie_vorige_volgende_morgen,"[vakantie, vorige, volgende, morgen, infuus, komende, weekend, last, spuit, ochtend]","[Komende vrijdag moet ik weer voor mijn infuus gaan., Als dat zo is, zou mijn infuus morgen door kunnen gaan., 4 aug. is 6 weken en 3 dagen na het vorige infuus, kan dat?]"
3,2,1373,2_huisarts_arts_dokter_ziekenhuis,"[huisarts, arts, dokter, ziekenhuis, mdl, contact, verpleegkundige, medische, medisch, reumatoloog]","[ik heb contact gehad met de huisarts., Ik heb woensdag een afspraak bij de huisarts., ik ben net bij de huisarts geweest.]"
4,3,1227,3_gepland_afspraak_staat_endoscopie,"[gepland, afspraak, staat, endoscopie, staan, poli, eind, vakantie, infuus, begin]","[De endoscopie staat gepland op 23 mei om 15:20., Op 11 mei staat er een endoscopie gepland., Op 23 oktober a.s. heb ik een afspraak bij gepland staan.]"
...,...,...,...,...,...
179,178,20,178_hoor_graag_geboorte_tips,"[hoor, graag, geboorte, tips, , , , , , ]","[Ik hoor graag van je,, Ik hoor graag van je,, Ik hoor graag van jullie,]"
180,179,20,179_risicogroep_behoor_val_risico,"[risicogroep, behoor, val, risico, groep, bso, weerstand, rivm, schrijft, verminderde]","[Val ik ook onder de risicogroep die nu geprikt wordt?, Behoor ik tot de risicogroep?, Of val ik niet meer onder de risicogroep.]"
181,180,20,180_darmkanker_bevolkingsonderzoek_meegedaan_deelname,"[darmkanker, bevolkingsonderzoek, meegedaan, deelname, onlangs, uitnodiging, oproep, deel, landelijk, zelfafnametest]","[heer , Onlangs een brief ontvangen om aan een bevolkingsonderzoek darmkanker mee te doen., Ik heb een uitnodiging gekregen voor het bevolkingsonderzoek naar darmkanker., Verder heb ik een vooraan..."
182,181,20,181_advies_adviseren_adviezen_aandachtspunten,"[advies, adviseren, adviezen, aandachtspunten, geven, miconazolnitraat, hydrocortison, vervolgrecept, richtlijnen, voorstel]","[Kan jullie a.u.b. advies?, Of heeft U een ander advies?, Hebben jullie een advies voor mij?]"


In [51]:
best_combination

(100, 6, 0.1)

In [50]:
coherence_score = calculate_c_v(best_model)
coherence_score

0.3705792198430164

In [52]:
# save best model 
# save best model
best_model.save("/workspace/mijnidbcoachnlp/saved_models/best_cv_score_st_model", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)

In [48]:
from sklearn.metrics import silhouette_score
from sklearn.datasets import fetch_20newsgroups
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
import numpy as np

# try calculating the silouette score
topics = best_model.topics_

# Generate `X` and `labels` only for non-outlier topics (as they are technically not clusters)
umap_embeddings = topic_model.umap_model.transform(embeddings)
indices = [index for index, topic in enumerate(topics) if topic != -1]
X = umap_embeddings[np.array(indices)]
labels = [topic for index, topic in enumerate(topics) if topic != -1]

# Calculate silhouette score
silhouette_score(X, labels)

0.42173073

In [63]:
# try visualize the topics
# settings for plotly
import plotly.io as pio
pio.renderers.default = "notebook"
pio.renderers.default = "browser" # option to open the plots in brower

In [55]:
# Visualize the inter-topic distance map
best_model.visualize_topics()

In [74]:
best_umap_model = best_model
best_umap_score = best_score

### 3.2 Fine-tune HDBSCAN hyperparameters

In [87]:
import itertools

# Define parameter ranges
range_min_cluster_size = [10, 20, 35, 50]
range_min_samples = [10, 20, 30, 50]

# Generate all possible combinations of the parameters
hdbscan_combinations = list(itertools.product(range_min_cluster_size, range_min_samples))

In [91]:
hdbscan_combinations

[(10, 10),
 (10, 20),
 (10, 30),
 (10, 50),
 (20, 10),
 (20, 20),
 (20, 30),
 (20, 50),
 (35, 10),
 (35, 20),
 (35, 30),
 (35, 50),
 (50, 10),
 (50, 20),
 (50, 30),
 (50, 50)]

In [79]:
import random
from joblib import Parallel, delayed

# tuning hdbscan parameters min_cluster_size and mean_samples
def tune_bertopic_hdbscan_random(embedding_model, vectorizer_model, embeddings, sentences, combinations, n_iter, best_score, best_model):
    """
    Randomized search over UMAP and HDBSCAN hyperparameters to find the best model based on coherence score.
    
    Args:
        embedding_model: The embedding model to use for topic modeling.
        vectorizer_model: The vectorizer model to use for tokenization.
        embeddings: The embeddings of the sentences.
        sentences: The sentences (documents) for topic modeling.
        n_iter: The number of random iterations to perform in the randomized search.
        
    Returns:
        best_topic_model: The best topic model based on coherence score.
        best_coherence_score: The best coherence score obtained during the search.
    """
    sample_combinations, untried_combinations = select_comb(n_iter, hdbscan_combinations)
    best_coherence_score = best_score
    best_topic_model = best_model

    # Randomized search loop
    for comb in sample_combinations:
        # Sample random UMAP parameters
        min_cluster_size = comb[0]
        min_samples = comb[1]
        # Tune UMAP and HDBSCAN
        umap_model = tune_umap(n_neighbors=100, n_components=6, min_dist=0.1)
        hdb_model = tune_hdb(min_cluster_size=min_cluster_size, min_samples=min_samples, cluster_selection_epsilon=0.0)

        # Build topic model
        current_topic_model = build_bertopic(
            embedding_model=embedding_model, 
            umap_model=umap_model, 
            cluster_model=hdb_model, 
            vectorizer_model=vectorizer_model, 
            embeddings=embeddings, 
            docs=sentences
        )
        
        # Calculate coherence score for the current model
        current_coherence_score = calculate_c_v(current_topic_model)
        print(f"Coherence score: {current_coherence_score} for HDBSCAN params: {min_cluster_size}, {min_samples}")
        
        # Check if the current model has a better coherence score
        if current_coherence_score > best_coherence_score:
            best_coherence_score = current_coherence_score
            best_topic_model = current_topic_model
            best_combination = (min_cluster_size, min_samples)
            
    return best_topic_model, best_coherence_score, best_combination, untried_combinations




In [80]:
# Example usage
best_model, best_score, best_combination, untried_combinations = tune_bertopic_hdbscan_random(embedding_model=embedding_model, vectorizer_model=vectorizer_model, embeddings=embeddings, sentences=sentences, n_iter=25, combinations=hdbscan_combinations, best_score=best_umap_score, best_model=best_umap_model)

2025-02-10 17:13:56,107 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-10 17:18:44,331 - BERTopic - Dimensionality - Completed ✓
2025-02-10 17:18:44,336 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-10 17:18:57,140 - BERTopic - Cluster - Completed ✓
2025-02-10 17:18:57,169 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-10 17:18:58,152 - BERTopic - Representation - Completed ✓
2025-02-10 17:19:37,776 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Coherence score: 0.3698040619335527 for HDBSCAN params: 35, 50


2025-02-10 17:25:11,683 - BERTopic - Dimensionality - Completed ✓
2025-02-10 17:25:11,688 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-10 17:25:19,206 - BERTopic - Cluster - Completed ✓
2025-02-10 17:25:19,231 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-10 17:25:20,299 - BERTopic - Representation - Completed ✓
2025-02-10 17:26:12,694 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Coherence score: 0.3658965537909055 for HDBSCAN params: 50, 5


2025-02-10 17:31:33,715 - BERTopic - Dimensionality - Completed ✓
2025-02-10 17:31:33,721 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-10 17:31:43,222 - BERTopic - Cluster - Completed ✓
2025-02-10 17:31:43,246 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-10 17:31:44,389 - BERTopic - Representation - Completed ✓
2025-02-10 17:32:45,795 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


...

Coherence score: 0.35508391508015086 for HDBSCAN params: 20, 30


2025-02-10 18:37:58,936 - BERTopic - Dimensionality - Completed ✓
2025-02-10 18:37:58,940 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-10 18:38:03,686 - BERTopic - Cluster - Completed ✓
2025-02-10 18:38:03,703 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-10 18:38:04,504 - BERTopic - Representation - Completed ✓
2025-02-10 18:38:57,853 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Coherence score: 0.36325580666109525 for HDBSCAN params: 10, 20


2025-02-10 18:43:15,182 - BERTopic - Dimensionality - Completed ✓
2025-02-10 18:43:15,186 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-10 18:43:22,119 - BERTopic - Cluster - Completed ✓
2025-02-10 18:43:22,141 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-10 18:43:23,366 - BERTopic - Representation - Completed ✓


ValueError: unable to interpret topic as either a list of tokens or a list of ids

In [92]:

untried_hdb_combinations = set(hdbscan_combinations) - set([(35, 50), (50, 5), (5, 20), (5, 20), (50, 10), (25, 5), (35, 20), (35, 10), (10, 10), (50, 30), (35, 5), (10, 50), (20, 30), (10, 20)])
untried_hdb_combinations

{(10, 30), (20, 10), (20, 20), (20, 50), (35, 30), (50, 20), (50, 50)}

In [82]:
print("Best HDBSCAN Parameters:", best_combination)
print("Best Coherence Score:", best_coherence_score)

Best HDBSCAN Parameters: (100, 6, 0.1)


NameError: name 'best_coherence_score' is not defined

In [83]:
umap_model = tune_umap(n_neighbors=100, n_components=6, min_dist=0.1)
hdb_model = tune_hdb(min_cluster_size=5, min_samples=10, cluster_selection_epsilon=0.0)
best_model = build_bertopic(
            embedding_model=embedding_model, 
            umap_model=umap_model, 
            cluster_model=hdb_model, 
            vectorizer_model=vectorizer_model, 
            embeddings=embeddings, 
            docs=sentences
        )

2025-02-10 19:05:15,746 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-10 19:09:26,529 - BERTopic - Dimensionality - Completed ✓
2025-02-10 19:09:26,533 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-10 19:09:33,758 - BERTopic - Cluster - Completed ✓
2025-02-10 19:09:33,783 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-10 19:09:34,984 - BERTopic - Representation - Completed ✓


In [86]:
coherence_score = calculate_c_v(best_model)
coherence_score

0.3719332611664019

In [84]:
best_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,20379,-1_medicatie_weet_last_gaat,"[medicatie, weet, last, gaat, klachten, krijg, tijd, gaan, goed, pijn]","[ik heb een vraag naar mijn reactie vorige week op Remicade ik heb ik nu iets meer last van mijn darmen last van krampen en stekkende pijn, niet elke dag maar af en toe., Er staat in mijn medicati..."
1,0,3364,0_bloed_prikken_bloedprikken_slijm,"[bloed, prikken, bloedprikken, slijm, bloeduitslagen, ontlasting, laten, geprikt, ingeleverd, inleveren]","[Afgelopen maandag heb ik bloed laten prikken., en , Ik heb woensdag bloed laten prikken., Twee weken geleden heb ik bloed laten prikken.]"
2,1,1373,1_huisarts_arts_dokter_mdl,"[huisarts, arts, dokter, mdl, ziekenhuis, contact, verpleegkundige, medische, medisch, reumatoloog]","[Ja bij mijn huisarts., Of had de huisarts dit moeten doen?, Bij de huisarts of in het ziekenhuis?]"
3,2,1227,2_gepland_afspraak_endoscopie_staat,"[gepland, afspraak, endoscopie, staat, staan, poli, eind, vakantie, scopie, begin]","[Er staat op dinsdag, 28 juli, met jou een afspraak gepland om 10.00u., Op 1 september staat voor mij een afspraak gepland bij ., Op 20 augustus staat een poli-afspraak gepland.]"
4,3,745,3_nieuw_recept_apotheek_sturen,"[nieuw, recept, apotheek, sturen, herhaal, nethol, puri, purinethol, uitschrijven, herhaalrecept]","[Graag zou ik een nieuw recept willen hebben ,voor de purenetol 50 mg graag sturen naar Apotheek van ,gijsen, Zou u voor mij een nieuw recept naar mijn apotheek kunnen sturen?, Kunt u een nieuw r..."
...,...,...,...,...,...
322,321,5,321_duidelijk_duidelijkheid_dacht_helemaal,"[duidelijk, duidelijkheid, dacht, helemaal, , , , , , ]","[Helemaal duidelijk., Dit is duidelijk., Alles is duidelijk.]"
323,322,5,322_aanpassingen_horen_betreft_kant,"[aanpassingen, horen, betreft, kant, medicatie, starten, hoor, medicijn, darmen, fijn]","[Ik hoor graag van u wat betreft de medicatie., Mochten er van u kant aanpassingen moeten komen in mijn medicijn gebruik, dan hoor ik dat graag., Ik hoor graag van jullie horen nu verder betreft m..."
324,323,5,323_stelt_gerust_alvast_reactie,"[stelt, gerust, alvast, reactie, moeite, dankjewel, bedankt, dank, goed, ]","[Bedankt voor je reactie alvast,, Alvast bedankt voor jullie reactie,, dankjewel voor de reactie is goed dat stelt me gerust]"
325,324,5,324_doorgezet_verstuurd_mailen_recept,"[doorgezet, verstuurd, mailen, recept, zorginstelling, apotheek, sturen, ziekenhuis, , ]","[Het recept moet verstuurd worden naar [APOTHEEK] ., Het recept mag doorgezet worden naar de apotheek in het [ZIEKENHUIS]., U kunt het recept Mailen naar [ZORGINSTELLING].]"
